# GUNDAM interface demo

This notebook configures GUNDAM through `gundam-interface`, initializes the fitter engine, evaluates the nominal likelihood, runs a minimization, and inspects the best-fit parameters.


In [1]:
nCpuThreads = 3
gundamLibPath = "/Users/nadrino/Documents/Work/Install/gundam/lib"
workDir = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024"
configPath = "configOA2024.yaml"
overrideList = [
    "override/v12ProdRun45.yaml",
    "override/onlyFlux5.yaml",
    "override/noEigen.yaml",
]
dataType = "Asimov"  # "Asimov", "Toy", or "RealData"
seed = 12345

In [2]:
import sys
import numpy as np
from pathlib import Path

# Prefer the local checkout when running this notebook before pip installation.
# If you install this package with pip, you can remove this block.
# User configuration
repoRoot = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
srcPath = repoRoot / "src"
if srcPath.exists() and str(srcPath) not in sys.path:
    sys.path.insert(0, str(srcPath))
# ~ end of this block

from gundam_interface import GundamLoader, GundamRuntime, GundamInterface


In [3]:
np.random.seed(seed)

runtime = GundamRuntime(
    loader=GundamLoader(gundamLibPath=gundamLibPath),
    workDir=workDir,
    nCpuThreads=nCpuThreads,
    configPath=configPath,
    overrideList=overrideList,
    dataType=dataType,
    randomSeed=seed,
)

runtime.toDict(includeConfigJsonString=False)


{'nCpuThreads': 3,
 'workDir': '/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024',
 'dataType': 'Asimov',
 'loader': {'moduleName': 'GUNDAM',
  'gundamLibPath': '/Users/nadrino/Documents/Work/Install/gundam/lib'},
 'randomSeed': 12345,
 'configPath': 'configOA2024.yaml',
 'overrideList': ['override/v12ProdRun45.yaml',
  'override/onlyFlux5.yaml',
  'override/noEigen.yaml']}

In [4]:
gundam = GundamInterface(runtime)
gundam.configure()

2026.06.30 15:41:49  INFO ConfigUtils: Reading config file: /Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/configOA2024.yaml
2026.06.30 15:41:49  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/v12ProdRun45.yaml"
2026.06.30 15:41:49  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/mc/filePathList
2026.06.30 15:41:49  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/data/0(name:"data")/filePathList
2026.06.30 15:41:49  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/onlyFlux5.yaml"
2026.06.30 15:41:49  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSetListConfig/0(name:"Flux Systematics")/isEnabled: true -> false
2026.06.30 15:41:49  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSe

In [5]:
runtime.logRedirector.debug = True
gundam.initialize()

GUNDAM native output is redirected to temporary log file (auto-deleted after execution): /var/folders/g0/chby37ds1dn73tq5rl8q0_640000gp/T/gundam_initialize_t2gc_mdv.log
GUNDAM native output is redirected to log file: /var/folders/g0/chby37ds1dn73tq5rl8q0_640000gp/T/gundam_initialize_t2gc_mdv.log
2026.06.30 15:41:50 ALERT ConfigUtils: /fitterEngineConfig/propagatorConfig: key "showEventBreakdownBeforePrior" was ignored while parsing config. Is is context dependent option?
2026.06.30 15:41:50  INFO ParameterSet: Stripping the matrix from fixed/disabled parameters...
2026.06.30 15:41:50  INFO ParameterSet: 5 effective parameters were defined in set: Flux Systematics
2026.06.30 15:41:50  INFO ParameterSet: Computing inverse of the stripped covariance matrix: 5x5
2026.06.30 15:41:50  INFO ParametersManager: 5 enabled parameters in Flux Systematics
2026.06.30 15:41:50  INFO ParametersManager: Total number of parameters: 5
2026.06.30 15:41:50  INFO ParametersManager: Building global covarianc

In [6]:
print(f"Initialized GUNDAM with {len(gundam.parameters)} active parameters")
for index, parameter in enumerate(gundam.parameters):
    print(
        f"{index:3d}: {parameter.name} "
        f"prior={parameter.prior:.8g} "
        f"step={parameter.stepSize:.8g} "
        f"value={parameter.value:.8g}"
    )

Initialized GUNDAM with 843 active parameters
  0: Flux Systematics/#0 prior=1 step=0.05924376 value=1
  1: Flux Systematics/#1 prior=1 step=0.052534918 value=1
  2: Flux Systematics/#2 prior=1 step=0.052933735 value=1
  3: Flux Systematics/#3 prior=1 step=0.051454976 value=1
  4: Flux Systematics/#4 prior=1 step=0.073819516 value=1
  5: Flux Systematics/#5 prior=1 step=0.087584986 value=1
  6: Flux Systematics/#6 prior=1 step=0.069835737 value=1
  7: Flux Systematics/#7 prior=1 step=0.049799122 value=1
  8: Flux Systematics/#8 prior=1 step=0.050205095 value=1
  9: Flux Systematics/#9 prior=1 step=0.070165897 value=1
 10: Flux Systematics/#10 prior=1 step=0.11426228 value=1
 11: Flux Systematics/#11 prior=1 step=0.097235633 value=1
 12: Flux Systematics/#12 prior=1 step=0.065676759 value=1
 13: Flux Systematics/#13 prior=1 step=0.071617626 value=1
 14: Flux Systematics/#14 prior=1 step=0.088904925 value=1
 15: Flux Systematics/#15 prior=1 step=0.076413478 value=1
 16: Flux Systematics/

In [7]:
nominalPhysicalValues = gundam.getParameterValues()
nominalLlh = gundam.evaluateLlh()

print(f"Nominal LLH: {nominalLlh:.8g}")
print("Nominal parameters:")
for name, physicalValue in zip(gundam.parameterNames, nominalPhysicalValues):
    print(f"  - {name}: physical={physicalValue:.8g}")


Nominal LLH: 1.1137326e-14
Nominal parameters:
  - Flux Systematics/#0: physical=1
  - Flux Systematics/#1: physical=1
  - Flux Systematics/#2: physical=1
  - Flux Systematics/#3: physical=1
  - Flux Systematics/#4: physical=1
  - Flux Systematics/#5: physical=1
  - Flux Systematics/#6: physical=1
  - Flux Systematics/#7: physical=1
  - Flux Systematics/#8: physical=1
  - Flux Systematics/#9: physical=1
  - Flux Systematics/#10: physical=1
  - Flux Systematics/#11: physical=1
  - Flux Systematics/#12: physical=1
  - Flux Systematics/#13: physical=1
  - Flux Systematics/#14: physical=1
  - Flux Systematics/#15: physical=1
  - Flux Systematics/#16: physical=1
  - Flux Systematics/#17: physical=1
  - Flux Systematics/#18: physical=1
  - Flux Systematics/#19: physical=1
  - Flux Systematics/#20: physical=1
  - Flux Systematics/#21: physical=1
  - Flux Systematics/#22: physical=1
  - Flux Systematics/#23: physical=1
  - Flux Systematics/#24: physical=1
  - Flux Systematics/#25: physical=1
 

In [8]:
bestFitLlh = gundam.minimize()
bestFitPhysicalValues = gundam.getParameterValues()
deltaLlh = bestFitLlh - nominalLlh

print(f"Best-fit LLH: {bestFitLlh:.8g}")
print(f"Delta LLH relative to nominal: {deltaLlh:.8g}")



2026.06.30 15:26:29  INFO MinimizerBase: ──────────────────────────────
2026.06.30 15:26:29  INFO MinimizerBase: Summary of the fit parameters:
2026.06.30 15:26:29  INFO MinimizerBase: ──────────────────────────────
2026.06.30 15:26:29  INFO MinimizerBase: Flux Systematics: 100 parameters
┌───────┬──────────┬──────────┬──────────┬──────────────┬────────────────┐
│ Title │ Starting │    Prior │   StdDev │       Limits │         Status │
├───────┼──────────┼──────────┼──────────┼──────────────┼────────────────┤
│    #0 │ 1.000000 │ 1.000000 │ 0.059244 │ [-inf, +inf] │ Gaussian Prior │
│    #1 │ 1.000000 │ 1.000000 │ 0.052535 │ [-inf, +inf] │ Gaussian Prior │
│    #2 │ 1.000000 │ 1.000000 │ 0.052934 │ [-inf, +inf] │ Gaussian Prior │
│    #3 │ 1.000000 │ 1.000000 │ 0.051455 │ [-inf, +inf] │ Gaussian Prior │
│    #4 │ 1.000000 │ 1.000000 │ 0.073820 │ [-inf, +inf] │ Gaussian Prior │
│    #5 │ 1.000000 │ 1.000000 │ 0.087585 │ [-inf, +inf] │ Gaussian Prior │
│    #6 │ 1.000000 │ 1.000000 │ 0.

In [9]:
print("Best-fit parameters:")
for name, physicalValue in zip(gundam.parameterNames, bestFitPhysicalValues):
    print(f"  - {name}: physical={physicalValue:.8g}")


Best-fit parameters:
  - Flux Systematics/#0: physical=1
  - Flux Systematics/#1: physical=1
  - Flux Systematics/#2: physical=1
  - Flux Systematics/#3: physical=1
  - Flux Systematics/#4: physical=1
  - Flux Systematics/#5: physical=1
  - Flux Systematics/#6: physical=1
  - Flux Systematics/#7: physical=1
  - Flux Systematics/#8: physical=1
  - Flux Systematics/#9: physical=1
  - Flux Systematics/#10: physical=1
  - Flux Systematics/#11: physical=1
  - Flux Systematics/#12: physical=1
  - Flux Systematics/#13: physical=1
  - Flux Systematics/#14: physical=1
  - Flux Systematics/#15: physical=1
  - Flux Systematics/#16: physical=1
  - Flux Systematics/#17: physical=1
  - Flux Systematics/#18: physical=1
  - Flux Systematics/#19: physical=1
  - Flux Systematics/#20: physical=1
  - Flux Systematics/#21: physical=1
  - Flux Systematics/#22: physical=1
  - Flux Systematics/#23: physical=1
  - Flux Systematics/#24: physical=1
  - Flux Systematics/#25: physical=1
  - Flux Systematics/#26: p

In [10]:
# Re-evaluate the likelihood at the best-fit point as a consistency check.
reevaluatedBestFitLlh = gundam.evaluateLlh(physicalValues=bestFitPhysicalValues)

print(f"Minimizer best-fit LLH: {bestFitLlh:.8g}")
print(f"Re-evaluated best-fit LLH: {reevaluatedBestFitLlh:.8g}")
print(f"Absolute difference: {abs(reevaluatedBestFitLlh - bestFitLlh):.8g}")


Minimizer best-fit LLH: -1.3624095e-13
Re-evaluated best-fit LLH: -1.3624095e-13
Absolute difference: 0
